## **Associative Classification ( CBA )**

Association rules can also be used to **classify**. In **associative classification** the rules mined have the form `condition ⟶ class` ( **class association rules, CARs** ), such as `flavanoids ≥ 2.88 AND alcohol ≥ 13.7 ⟶ class_0`. The classifier is an **ordered list of the best rules** plus a **default class**, and a new case receives the class of the **first rule** that matches it.

This notebook implements **CBA** ( *Classification Based on Associations*, Liu, Hsu and Ma, 1998 ) from scratch: the rules are mined with the same level-wise search of Apriori, and the list is built with the **database coverage** method. The result is evaluated with **leakage-free cross-validation** against ordinary classifiers.

**Notebooks of this series:** 1. Apriori · 2. FP-Growth · 3. Eclat · 4. Comparing the Algorithms · 5. Association Rules ( Generation, Reduction and Evaluation ) · 6. Interestingness Measures · 7. Sequential Patterns ( SPADE ) · 8. High-Utility Itemsets · 9. Associative Classification ( CBA )

### **0. Environment Setup**

In [1]:
#Import Libs
import sklearn
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

In [2]:
#Define Colors Palette and Standards for Illustrations
RED, DARK, GRAY, LIGHT, SOFT = "#CD2027", "#404040", "#7F7F7F", "#D9D9D9", "#E9A0A3"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
    "axes.prop_cycle": cycler(color=[RED, DARK, SOFT, GRAY]),
})
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)

In [3]:
#Helper Functions: Evaluation Reports ( Confusion Matrix, Metrics, ROC and Precision-Recall )
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, log_loss, matthews_corrcoef,
                             confusion_matrix, roc_curve, precision_recall_curve)

def plot_confusion(cm, labels, ax, title="Confusion matrix"):
    ax.imshow(cm, cmap="Reds"); ax.grid(False)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45 if len(labels) > 4 else 0, ha="right" if len(labels) > 4 else "center")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=10 if len(labels) > 4 else 13,
                    color="white" if cm[i, j] > cm.max() / 2 else DARK)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)

def report_binary(y_true, y_pred, y_score, name="model", labels=("negative", "positive"), plot=True):
    """Confusion matrix, ROC and Precision-Recall curves + a Series with the main metrics (positive class = 1)."""
    y_true, y_pred, y_score = np.asarray(y_true), np.asarray(y_pred), np.asarray(y_score)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "precision": precision_score(y_true, y_pred, zero_division=0), "recall": recall_score(y_true, y_pred),
               "F1": f1_score(y_true, y_pred), "MCC": matthews_corrcoef(y_true, y_pred),
               "ROC AUC": roc_auc_score(y_true, y_score), "PR AUC": average_precision_score(y_true, y_score)}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
        plot_confusion(confusion_matrix(y_true, y_pred), list(labels), axes[0], f"Confusion matrix – {name}")
        fpr, tpr, _ = roc_curve(y_true, y_score)
        axes[1].plot(fpr, tpr, color=RED, lw=2.4, label=f"AUC = {metrics['ROC AUC']:.3f}"); axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--")
        axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate"); axes[1].set_title("ROC curve"); axes[1].legend(loc="lower right")
        p, r, _ = precision_recall_curve(y_true, y_score)
        axes[2].plot(r, p, color=DARK, lw=2.4, label=f"AP = {metrics['PR AUC']:.3f}"); axes[2].axhline(y_true.mean(), color=GRAY, ls="--")
        axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision"); axes[2].set_title("Precision–Recall curve"); axes[2].legend(loc="lower left")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)

def report_multiclass(y_true, y_pred, y_proba, labels, name="model", plot=True):
    """Confusion matrix, one-vs-rest ROC curves and per-class F1 + a Series with the main metrics."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    k = len(labels)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "macro precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
               "macro recall": recall_score(y_true, y_pred, average="macro"), "macro F1": f1_score(y_true, y_pred, average="macro"),
               "MCC": matthews_corrcoef(y_true, y_pred), "ROC AUC (ovr)": roc_auc_score(y_true, y_proba, multi_class="ovr"),
               "log loss": log_loss(y_true, y_proba, labels=list(range(k)))}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.1, 1, 1]})
        plot_confusion(confusion_matrix(y_true, y_pred, labels=list(range(k))), list(labels), axes[0], f"Confusion matrix – {name}")
        cmap = plt.get_cmap("tab10")
        for c in range(k):
            fpr, tpr, _ = roc_curve((y_true == c).astype(int), y_proba[:, c])
            axes[1].plot(fpr, tpr, color=cmap(c), lw=1.6, label=f"{labels[c]}")
        axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--"); axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
        axes[1].set_title(f"ROC one-vs-rest (macro AUC = {metrics['ROC AUC (ovr)']:.3f})"); axes[1].legend(fontsize=7, ncol=2 if k > 5 else 1, loc="lower right")
        f1s = f1_score(y_true, y_pred, average=None)
        axes[2].bar(range(k), f1s, color=RED, width=0.65); axes[2].set_xticks(range(k)); axes[2].set_xticklabels(labels, rotation=45 if k > 4 else 0, ha="right" if k > 4 else "center")
        axes[2].set_ylim(0, 1.05); axes[2].set_ylabel("F1"); axes[2].set_title("F1 per class")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)
def annotate_heatmap(ax, mat, fmt="{:.2f}"):
    """Write the values on an imshow heat map with a readable text color."""
    mat = np.asarray(mat, dtype=float)
    lo, hi = np.nanmin(mat), np.nanmax(mat)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            ax.text(j, i, "–" if np.isnan(v) else fmt.format(v), ha="center", va="center", color="white" if (not np.isnan(v) and v > (lo + hi) / 2) else DARK)

### **1. Dataset: Wine ( 3 Classes, 13 Numeric Features )**

Rules need **items**, so the numeric features must be **discretized**. Here each feature is split into 4 **quantile bins** ( `feature < t1`, `t1 ≤ feature < t2`, … ), and every bin is an item.

Attention! The bins are learned from the data, so they must be learned **only from the training part of each fold**. Otherwise the test cases influence the items ( data leakage ). The estimator of section 5 does it inside `fit`, and section 7 measures what happens when it is done wrongly.

In [ ]:
#Imports for Associative Classification and Load the Wine Dataset
import time
from itertools import combinations
from collections import defaultdict
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.datasets import load_wine
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_val_predict, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline

wine = load_wine(as_frame=True)
X_wine, y_wine = wine.data, wine.target.to_numpy()
class_names = [str(c) for c in wine.target_names]
print("cases:", X_wine.shape[0], "| features:", X_wine.shape[1], "| classes:", dict(zip(class_names, np.bincount(y_wine))))
X_wine.describe().T[["mean", "std", "min", "max"]].head(6)

In [ ]:
#Discretization into Items: Bins of One Feature ( Quantile Strategy ) and Support of Each Item
def fit_discretizer(X, n_bins=4):
    kbd = KBinsDiscretizer(n_bins, encode="onehot-dense", strategy="quantile", subsample=None).fit(X)
    names, feature_of = [], []
    for f, feature in enumerate(X.columns):
        n_f, edges = kbd.n_bins_[f], kbd.bin_edges_[f]
        for k in range(n_f):
            lo, hi = edges[k], edges[k + 1]
            names.append(f"{feature} < {hi:.3g}" if k == 0 else f"{feature} >= {lo:.3g}" if k == n_f - 1 else f"{lo:.3g} <= {feature} < {hi:.3g}")
            feature_of.append(f)
    return kbd, names, np.array(feature_of)

kbd_demo, item_names_demo, feature_of_demo = fit_discretizer(X_wine)
B_demo = kbd_demo.transform(X_wine) > 0
print("items:", B_demo.shape[1], "( 13 features x 4 bins ) | items per case:", B_demo.sum(axis=1).mean())
class_profile = pd.DataFrame(B_demo[:, [i for i, n in enumerate(item_names_demo) if n.startswith("flavanoids")]], columns=[n for n in item_names_demo if n.startswith("flavanoids")]).groupby(wine.target.map(dict(enumerate(class_names))).to_numpy()).mean()
class_profile.round(2)

### **2. Mining Class Association Rules ( CARs )**

A rule `condition ⟶ class` has **support** = share of the training cases that match the condition **and** have the class, and **confidence** = share of the cases matching the condition that have the class. The search is the level-wise Apriori with one change: a condition is *frequent* if at least **one class** reaches the minimum support together with it ( still anti-monotone ). Two bins of the same feature can never occur together, so those joins are skipped.

In [ ]:
#Mining Class Association Rules ( Level-Wise Search on Boolean Matrices )
def mine_cars(B, y, n_classes, min_support, min_confidence, max_len, feature_of):
    n = len(y)
    min_count = int(np.ceil(min_support * n))
    Y = np.stack([y == c for c in range(n_classes)], axis=1)
    cond_counts = lambda mask: (mask[:, None] & Y).sum(axis=0)                           # cases matching the condition, by class
    level = {(j,): B[:, j] for j in range(B.shape[1]) if cond_counts(B[:, j]).max() >= min_count}
    rules, per_level, size = [], [], 1
    while level:
        n_rules = 0
        for key, mask in level.items():
            counts, total = cond_counts(mask), int(mask.sum())
            for c in range(n_classes):
                if counts[c] >= min_count and counts[c] / total >= min_confidence:
                    rules.append({"cond": key, "cls": c, "count": int(counts[c]), "total": total, "support": counts[c] / n, "confidence": counts[c] / total, "mask": mask})
                    n_rules += 1
        per_level.append({"size": size, "frequent conditions": len(level), "class association rules": n_rules})
        if size == max_len:
            break
        groups = defaultdict(list)
        for key in level:
            groups[key[:-1]].append(key)
        nxt = {}
        for keys in groups.values():
            keys.sort()
            for a, c in combinations(keys, 2):
                if feature_of[a[-1]] == feature_of[c[-1]]:
                    continue                                                              # two bins of the same feature never co-occur
                cand = a + (c[-1],)
                if any(cand[:i] + cand[i + 1:] not in level for i in range(len(cand) - 2)):
                    continue                                                              # Apriori pruning
                mask2 = level[a] & B[:, c[-1]]
                if cond_counts(mask2).max() >= min_count:
                    nxt[cand] = mask2
        level, size = nxt, size + 1
    return rules, pd.DataFrame(per_level)

#Exploration Only ( Whole Dataset ): The Evaluation of Section 6 Does NOT Use These Rules
cars_demo, levels_demo = mine_cars(B_demo, y_wine, 3, min_support=0.10, min_confidence=0.8, max_len=3, feature_of=feature_of_demo)
print("class association rules ( support ≥ 10%, confidence ≥ 80% ):", len(cars_demo))
display(levels_demo)
sorted_demo = sorted(cars_demo, key=lambda r: (-r["confidence"], -r["support"], len(r["cond"])))[:6]
pd.DataFrame([{"IF": " AND ".join(item_names_demo[j] for j in r["cond"]), "THEN": class_names[r["cls"]], "support": r["support"], "confidence": r["confidence"]} for r in sorted_demo]).round(3)

### **3. CBA-CB: Selecting the Rules by Database Coverage**

Hundreds of CARs overlap heavily. CBA builds the classifier in four steps:

1. **Sort** the rules by confidence ( descending ), then support ( descending ), then size ( shorter first )
2. For each rule, in order: keep it only if it **classifies correctly at least one training case that is still uncovered**; then remove **all the cases it matches**
3. After each kept rule, the **default class** is the majority class of the remaining cases, and the **total training errors** ( errors of the kept rules + errors of the default class ) are recorded
4. **Cut** the list at the rule with the **fewest total errors**; the rules after it are discarded

In [ ]:
#CBA-CB ( Database Coverage ) and Rule Application
def sort_rules(rules):
    return sorted(rules, key=lambda r: (-r["confidence"], -r["support"], len(r["cond"]), r["cond"], r["cls"]))

def select_cba(rules, y, n_classes):
    n = len(y)
    remaining, selected, trace, rule_errors = np.ones(n, bool), [], [], 0
    global_counts = np.bincount(y, minlength=n_classes)
    for r in sort_rules(rules):
        m = r["mask"] & remaining
        if not (m & (y == r["cls"])).any():
            continue                                                                      # it would not classify correctly any uncovered case
        selected.append(r)
        remaining &= ~m
        rule_errors += int((m & (y != r["cls"])).sum())
        rest = np.bincount(y[remaining], minlength=n_classes)
        trace.append({"rules": len(selected), "rule errors": rule_errors, "default errors": int(rest.sum() - rest.max()), "remaining": int(rest.sum()), "rest": rest})
        if not remaining.any():
            break
    for t in trace:
        t["total errors"] = t["rule errors"] + t["default errors"]
    if not trace:
        return [], int(global_counts.argmax()), global_counts, pd.DataFrame()
    best = min(range(len(trace)), key=lambda i: (trace[i]["total errors"], i))             # first minimum of the total training errors
    rest = trace[best]["rest"] if trace[best]["rest"].sum() else global_counts
    return selected[:best + 1], int(rest.argmax()), rest, pd.DataFrame(trace).drop(columns="rest")

def apply_rules(rules, B, n_classes):
    first = -np.ones(len(B), int)                                                         # index of the first rule that matches each case ( -1 = none )
    best = np.zeros((len(B), n_classes))                                                  # highest confidence of the matching rules, by class
    for idx, r in enumerate(rules):
        m = B[:, list(r["cond"])].all(axis=1)
        first[m & (first < 0)] = idx
        best[m, r["cls"]] = np.maximum(best[m, r["cls"]], r["confidence"])
    return first, best

rules_demo, default_demo, rest_demo, trace_demo = select_cba(cars_demo, y_wine, 3)
print(f"class association rules: {len(cars_demo)} -> selected by database coverage: {len(trace_demo)} -> final classifier after the cut: {len(rules_demo)} rules + default class '{class_names[default_demo]}'")
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(trace_demo["rules"], trace_demo["total errors"], color=RED, marker="o", ms=3, label="total training errors")
ax.plot(trace_demo["rules"], trace_demo["default errors"], color=GRAY, ls="--", label="errors of the default class")
ax.axvline(len(rules_demo), color=DARK, ls=":"); ax.set_xlabel("number of rules kept"); ax.set_ylabel("training errors"); ax.set_title("Where CBA cuts the rule list"); ax.legend()
plt.tight_layout(); plt.show()

### **4. Quality Metrics Plan**

The classifier is evaluated **only on cases that were not used to discretize, mine or select the rules**: stratified 5-fold cross-validation, with the whole pipeline ( bins → rules → rule list ) refitted in each fold. The scores are compared with the **majority class** ( a model that learns nothing ), a **shallow decision tree** ( also a rule-based model ), **Gaussian Naive Bayes** and **logistic regression** ( which use the raw numeric values ). The metrics are the generic ones of the helpers above: accuracy, balanced accuracy, macro F1, MCC, one-vs-rest ROC AUC and log loss.

### **5. A scikit-learn Estimator**

To use the cross-validation tools of scikit-learn **without leakage**, the whole procedure is wrapped in an estimator: the discretizer, the mining and the selection happen inside `fit`. `predict` uses the **first matching rule** ( the rule of CBA ); `predict_proba` is an auxiliary score for ROC AUC and log loss: the first rule gives its Laplace-corrected confidence to its class, and the remaining probability is shared by the other classes in proportion to the confidence of the other matching rules.

In [ ]:
#CBA as a scikit-learn Estimator ( Discretization, Mining and Rule Selection Inside fit )
class CBAClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, min_support=0.10, min_confidence=0.8, max_len=3, n_bins=4):
        self.min_support, self.min_confidence, self.max_len, self.n_bins = min_support, min_confidence, max_len, n_bins

    def fit(self, X, y):
        X = pd.DataFrame(X)
        self.classes_, y_idx = np.unique(y, return_inverse=True)
        k = len(self.classes_)
        self.kbd_, self.item_names_, feature_of = fit_discretizer(X, self.n_bins)                 # bins learned from the training data only
        cars, self.level_stats_ = mine_cars(self.kbd_.transform(X) > 0, y_idx, k, self.min_support, self.min_confidence, self.max_len, feature_of)
        self.n_cars_ = len(cars)
        self.rules_, self.default_, rest, self.trace_ = select_cba(cars, y_idx, k)
        self.default_proba_ = (rest + 1) / (rest.sum() + k)
        return self

    def _items(self, X):
        return self.kbd_.transform(pd.DataFrame(X)) > 0

    def predict(self, X):
        first, _ = apply_rules(self.rules_, self._items(X), len(self.classes_))
        rule_class = np.array([r["cls"] for r in self.rules_] + [self.default_])               # index -1 = default class
        return self.classes_[rule_class[first]]

    def predict_proba(self, X):
        k = len(self.classes_)
        first, best = apply_rules(self.rules_, self._items(X), k)
        proba = np.tile(self.default_proba_, (len(first), 1))
        for i in np.flatnonzero(first >= 0):
            r = self.rules_[first[i]]
            p_first = (r["count"] + 1) / (r["total"] + k)
            others = best[i] + 1e-3
            others[r["cls"]] = 0
            proba[i] = others / others.sum() * (1 - p_first)
            proba[i, r["cls"]] = p_first
        return proba

t0 = time.perf_counter()
cba_full = CBAClassifier().fit(X_wine, y_wine)
print(f"fit on the whole dataset: {time.perf_counter() - t0:.2f}s | CARs mined: {cba_full.n_cars_} | rules in the classifier: {len(cba_full.rules_)} | default class: {class_names[cba_full.classes_[cba_full.default_]]}")

### **6. Evaluation: 5-Fold Cross-Validation vs Other Classifiers**

In [ ]:
#Cross-Validated Predictions of All the Models ( Same Folds for Everyone )
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models = {"CBA ( association rules )": CBAClassifier(),
          "Decision tree ( depth 4 )": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
          "Gaussian Naive Bayes": GaussianNB(),
          "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
          "Majority class": DummyClassifier(strategy="prior")}
cv_pred, cv_proba, rows = {}, {}, []
for name, model in models.items():
    cv_pred[name] = cross_val_predict(model, X_wine, y_wine, cv=cv5)
    cv_proba[name] = cross_val_predict(model, X_wine, y_wine, cv=cv5, method="predict_proba")
    rows.append(report_multiclass(y_wine, cv_pred[name], cv_proba[name], class_names, name=name, plot=False))
cba_name = "CBA ( association rules )"
print("share of the cases where the most probable class of CBA equals the class of the first rule:", round((cv_proba[cba_name].argmax(axis=1) == cv_pred[cba_name]).mean(), 3))
pd.DataFrame(rows)

In [ ]:
#Detailed Report of CBA ( Confusion Matrix, ROC One-vs-Rest and F1 by Class )
report_multiclass(y_wine, cv_pred[cba_name], cv_proba[cba_name], class_names, name="CBA ( cross-validated )")

In [ ]:
#Repeated Cross-Validation ( 5 Folds x 5 Repetitions ): Mean and Standard Deviation of the Accuracy and of the Macro F1
from sklearn.model_selection import cross_validate
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
rep_rows = []
for name, model in models.items():
    res = cross_validate(model, X_wine, y_wine, cv=cv_rep, scoring=["accuracy", "f1_macro"])
    rep_rows.append({"model": name, "accuracy ( mean )": res["test_accuracy"].mean(), "accuracy ( std )": res["test_accuracy"].std(),
                     "macro F1 ( mean )": res["test_f1_macro"].mean(), "macro F1 ( std )": res["test_f1_macro"].std()})
pd.DataFrame(rep_rows).set_index("model").round(3)

### **7. Data Leakage: Discretizing Before the Split**

What if the bins are computed on **all** the cases before the cross-validation? The test cases then take part in the choice of the cut points. The loop below runs the **same** procedure in the two ways. The first one must reproduce the scores of the estimator ( a sanity check of the implementation ).

In [ ]:
#Same Procedure, Discretizing Inside the Folds ( Correct ) vs Before the Split ( Leakage )
def cv_cba_manual(X, y, bins_inside_folds, cv, **params):
    scores, coverage = [], []
    if not bins_inside_folds:
        kbd_all, _, feature_of_all = fit_discretizer(X, 4)
        B_all = kbd_all.transform(X) > 0
    for tr, te in cv.split(X, y):
        if bins_inside_folds:
            kbd, _, feature_of = fit_discretizer(X.iloc[tr], 4)
            B_tr, B_te = kbd.transform(X.iloc[tr]) > 0, kbd.transform(X.iloc[te]) > 0
        else:
            B_tr, B_te, feature_of = B_all[tr], B_all[te], feature_of_all
        cars, _ = mine_cars(B_tr, y[tr], 3, params["min_support"], params["min_confidence"], 3, feature_of)
        rules, default, _, _ = select_cba(cars, y[tr], 3)
        first, _ = apply_rules(rules, B_te, 3)
        pred = np.array([r["cls"] for r in rules] + [default])[first]
        scores.append((pred == y[te]).mean()); coverage.append((first >= 0).mean())
    return np.array(scores), np.array(coverage)

inside, cov_inside = cv_cba_manual(X_wine, y_wine, True, cv_rep, min_support=0.10, min_confidence=0.8)
leaky, cov_leaky = cv_cba_manual(X_wine, y_wine, False, cv_rep, min_support=0.10, min_confidence=0.8)
estimator_scores = cross_val_score(CBAClassifier(), X_wine, y_wine, cv=cv_rep, scoring="accuracy")
print("sanity check: manual loop with bins inside the folds == scikit-learn estimator:", np.allclose(inside, estimator_scores))
print(f"bins inside the folds ( correct ): accuracy {inside.mean():.4f} ± {inside.std():.4f} | cases classified by a rule ( not by the default class ): {cov_inside.mean():.1%}")
print(f"bins before the split ( leakage ): accuracy {leaky.mean():.4f} ± {leaky.std():.4f} | cases classified by a rule: {cov_leaky.mean():.1%}")
print(f"difference ( leakage - correct ): {leaky.mean() - inside.mean():+.4f} | standard deviation across the folds: {inside.std():.4f}")

Attention! With **unsupervised** binning ( quantiles ) the labels are not used to choose the cut points, so in this dataset the difference between the two procedures is **within the noise of the cross-validation** ( the leaky version is not better ). That does not make the shortcut safe: the original CBA uses a **supervised** discretization ( entropy / MDL ), where the labels decide the cut points and the leakage can be large. Learn the bins **inside each training fold** as a rule.

### **8. Sensitivity to the Parameters**

In [ ]:
#Grid: min_support x min_confidence ( 5-Fold CV Accuracy, CARs Mined and Rules in the Final Classifier )
supports, confidences = [0.05, 0.10, 0.15, 0.20], [0.6, 0.7, 0.8, 0.9]
acc_grid, cars_grid, rules_grid = (np.zeros((4, 4)) for _ in range(3))
for i, ms in enumerate(supports):
    for j, mc in enumerate(confidences):
        model = CBAClassifier(min_support=ms, min_confidence=mc)
        acc_grid[i, j] = cross_val_score(model, X_wine, y_wine, cv=cv5).mean()
        full = model.fit(X_wine, y_wine)
        cars_grid[i, j], rules_grid[i, j] = full.n_cars_, len(full.rules_)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.9))
for ax, mat, title, fmt in zip(axes, [acc_grid, cars_grid, rules_grid], ["CV accuracy", "class association rules mined", "rules in the final classifier"], ["{:.3f}", "{:.0f}", "{:.0f}"]):
    ax.imshow(mat, cmap="Reds"); ax.grid(False)
    ax.set_xticks(range(4)); ax.set_xticklabels(confidences); ax.set_yticks(range(4)); ax.set_yticklabels(supports)
    annotate_heatmap(ax, mat, fmt); ax.set_xlabel("min_confidence"); ax.set_ylabel("min_support"); ax.set_title(title)
plt.tight_layout(); plt.show()

Attention! This grid shows the **sensitivity** of the method. Do not pick the best cell and report its score: the same folds would be used to choose and to evaluate. To tune the parameters, use **nested cross-validation**.

In [ ]:
#Effect of the Maximum Length of the Conditions ( min_support 10%, min_confidence 80% )
rows = []
for L in [1, 2, 3, 4]:
    model = CBAClassifier(max_len=L)
    t0 = time.perf_counter(); res = cross_val_score(model, X_wine, y_wine, cv=cv_rep); elapsed = time.perf_counter() - t0
    full = model.fit(X_wine, y_wine)
    rows.append({"max conditions per rule": L, "CV accuracy": res.mean(), "CARs mined": full.n_cars_, "rules in the classifier": len(full.rules_), "time of 25 CV fits (s)": elapsed})
pd.DataFrame(rows).round(3)

### **9. The Final Classifier Is Readable**

In [ ]:
#The Ordered Rule List of the Classifier Fitted on All the Cases ( First Matching Rule Wins )
def describe_rules(model):
    rows = [{"#": i + 1, "IF": " AND ".join(model.item_names_[j] for j in r["cond"]), "THEN": class_names[model.classes_[r["cls"]]],
             "support": r["support"], "confidence": r["confidence"]} for i, r in enumerate(model.rules_)]
    rows.append({"#": "default", "IF": "no rule matches", "THEN": class_names[model.classes_[model.default_]], "support": np.nan, "confidence": np.nan})
    return pd.DataFrame(rows)

rule_list = describe_rules(cba_full)
print("rules:", len(rule_list) - 1, "| rules by class:", rule_list.iloc[:-1]["THEN"].value_counts().to_dict())
rule_list.head(10).round(3)

### **10. Takeaways**

* **Rules as a classifier:** CARs `condition ⟶ class` + ordering by confidence + database coverage + default class = a model that a person can **read, audit and edit**
* **It is the same engine of Apriori:** frequent conditions are mined level by level with the same pruning; only the counting by class and the selection step are new
* **Evaluation:** the whole pipeline ( bins, rules, list ) was refitted inside each fold, and compared with a no-information baseline and with other classifiers on the same folds. In this dataset CBA is far above the majority class and in the range of the shallow decision tree, **but below logistic regression and Naive Bayes**, which use the raw numeric values: discretizing into 4 bins loses information. Associative classification is chosen for **interpretability**, not for the highest accuracy
* **Leakage:** discretization is a learned transformation; learn it inside each training fold. Section 7 compares both procedures: with quantile bins the difference is within the noise, with supervised bins it would not be
* **Parameters:** a low min_support multiplies the rules ( section 8 ) without improving the accuracy, and conditions with 2 items improve on single-item conditions while a third item adds almost nothing ( maximum length table ); the parameter grid is a sensitivity analysis, not a tuning procedure
* Related methods: **CMAR** ( several rules vote, weighted chi-square ), **CPAR** ( rules learned by a FOIL-style greedy search ), **CBA2** ( different min_support by class, for imbalanced classes ), and **Bayesian rule lists** and **CORELS** ( optimal rule lists )